### **2. Stitch로 디자인하고 React 프로젝트로 Firebase 배포하기**

**활동 목표:** 구글의 웹 디자인 서비스 'Stitch'로 간단한 CRM 웹 앱 UI를 디자인하고, 그 코드를 React 프로젝트로 변환하여 Firebase로 배포합니다.

**실습 단계:**

#### **Step 1. Stitch로 CRM 웹 앱 UI 디자인하기**

1.  **[stitch.withgoogle.com](https://stitch.withgoogle.com/)** 에 접속하여 새 프로젝트를 시작합니다.

2.  아래 시나리오에 맞게 UI를 디자인해 보세요. Stitch는 직관적인 도구이므로, 자유롭게 요소를 추가하며 만들어보세요.

    > **[디자인 시나리오]**

    > react-router-dom을 사용하는 멀티 페이지 CRM 대시보드 UI를 React 코드로 만들어줘.

    > 전체 구조: 상단 헤더와 좌측 사이드바는 모든 페이지에서 고정으로 보여줘.

    > 사이드바 메뉴: '대시보드', '고객 목록' 두 개의 메뉴가 있고, 각 메뉴를 클릭하면 해당 페이지로 이동해야 해.

    > / (대시보드 페이지): "환영합니다!" 메시지와 몇 가지 요약 카드를 보여줘.

    > /customers (고객 목록 페이지): 고객 이름과 이메일이 담긴 테이블(표)을 보여줘.

    > 라우터 설정이 포함된 App.jsx와 각 페이지 컴포넌트 코드를 모두 제공해줘.

3.  디자인이 완성되면, **React 코드로 내보내기(Export to React)** 기능을 사용해 생성된 코드를 모두 복사합니다.


#### **Step 2. Colab에 React 프로젝트 생성하고 코드 심기**

1.  아래 명령어를 실행하여 Colab에 `my-crm-app`이라는 이름의 React 프로젝트를 생성합니다. `Vite`라는 최신 도구를 사용하여 매우 빠르게 만들 수 있습니다.

In [ ]:
!npm create vite@latest my-crm-app -- --template react

!cd my-crm-app && npm install react-router-dom

2.  Stitch에서 복사한 React 코드를, 방금 만든 프로젝트의 핵심 파일인 `App.jsx`에 덮어쓰겠습니다.

In [ ]:
%%writefile my-crm-app/src/App.jsx
// 여기에 Stitch에서 복사한 React 코드를 붙여넣으세요.

3.  기본 CSS 파일의 내용을 비워 깔끔하게 만듭니다.

In [ ]:
%%writefile my-crm-app/src/index.css
/* 기존 내용을 모두 지웁니다. */

4. 이제 최종 배포를 하기 전에, 우리의 React 앱이 Colab 환경에서 잘 작동하는지 실시간으로 확인해 보겠습니다. 아래 코드를 실행하여 React 개발 서버를 켜고, ngrok으로 공개 테스트 주소를 만드세요.

In [ ]:
# ngrok 설치 및 토큰 설정
!pip install pyngrok
from pyngrok import ngrok
import threading
import os

# ngrok 대시보드(https://dashboard.ngrok.com/get-started/your-authtoken)에서 토큰을 복사해오세요.
ngrok.set_auth_token("YOUR_NGROK_AUTHTOKEN")

# Vite 개발 서버를 백그라운드에서 실행 (Vite 기본 포트는 5173)
# --host 옵션으로 외부 접속을 허용합니다.
server_thread = threading.Thread(
    target=lambda: os.system("cd my-crm-app && npm run dev -- --host --port 5173")
)
server_thread.daemon = True
server_thread.start()

# ngrok으로 5173 포트에 대한 터널을 생성
public_url = ngrok.connect(5173)
print(f"🚀 React 앱 테스트 URL: {public_url}")


5. 출력된 React 앱 테스트 URL을 클릭하여 새 탭에서 앱을 여세요. 사이드바의 메뉴들을 클릭하며 페이지 이동(라우팅)이 잘 되는지 직접 테스트해 보세요!

#### **Step 3. Firebase에 React 프로젝트 배포하기**

1.  **Firebase 로그인:** 아래 코드를 실행하고, 안내에 따라 구글 계정으로 로그인하여 Firebase 사용 권한을 얻습니다.

In [ ]:
# Colab 환경에서 구글 계정으로 로그인합니다.
!curl -sL https://firebase.tools | bash

!firebase login

2.  **프로젝트 설정 및 빌드:**
      * [Firebase 콘솔](https://console.firebase.google.com/)에 접속하여 새 프로젝트를 만들고, **프로젝트 ID**를 복사해오세요. (예: `my-crm-project-a1b2c`)
      * 아래 코드 셀들을 순서대로 실행하여 배포 설정을 완료하고, React 프로젝트를 웹사이트용 정적 파일로 변환(빌드)합니다.

In [ ]:
# ⚠️ 아래 따옴표 안에 본인의 Firebase 프로젝트 ID를 붙여넣으세요.
FIREBASE_PROJECT_ID = "YOUR-PROJECT-ID"

In [ ]:
# React 프로젝트 폴더로 이동하여 필요한 라이브러리를 설치하고 프로젝트를 빌드합니다.
%cd my-crm-app
!npm install
!npm run build

3.  **배포 설정 파일 생성 및 배포 실행:**

In [ ]:
# .firebaserc 파일 생성: 기본 프로젝트를 지정합니다.
%%writefile .firebaserc
{
    "projects": {
    "default": "''' + FIREBASE_PROJECT_ID + '''"
    }
}

In [ ]:
# firebase.json 파일 생성: 배포할 폴더(React 빌드 결과물인 'dist')를 지정합니다.
%%writefile firebase.json
{
    "hosting": {
    "public": "my-crm-app/dist",
    "ignore": [
        "firebase.json",
        "**/.*",
        "**/node_modules/**"
    ],
    "rewrites": [
        {
        "source": "**",
        "destination": "/index.html"
        }
    ]
    }
}

4.  **드디어 배포\!** 아래 명령어를 실행하여 여러분의 CRM 웹 앱을 전 세계에 공개하세요.

In [ ]:
# in your deployment command
%cd ..
!firebase deploy --project $FIREBASE_PROJECT_ID --only hosting

5.  **결과 확인:** 배포가 완료되면 출력되는 **Hosting URL**을 클릭하여, Stitch로 디자인한 여러분의 React 웹 앱을 확인하세요!

### [추가] Step 4. [심화] Firebase 콘솔에서 Gemini 사용하기

배포가 끝났으니, 이제 Firebase에 내장된 Gemini를 사용해 간단한 백엔드 코드를 만들어 봅시다.

1. Firebase 콘솔 에 접속하여 여러분의 프로젝트를 선택하고, 왼쪽 메뉴에서 **빌드(Build) -> 함수(Functions)**로 이동하세요.

2. 'Generate function with Gemini' 또는 **Gemini 아이콘(✨)**이 붙어있는 버튼을 찾아 클릭합니다.

3. 나타나는 입력창에 아래와 같이 요청해보세요.

> **[프롬프트 예시]**
> "새로운 사용자가 Firebase 인증을 통해 가입할 때마다, 해당 사용자의 이메일을 Functions 로그에 출력하는 간단한 자바스크립트 Cloud Function을 만들어줘."
결과 확인: Gemini가 트리거, 런타임 설정, 그리고 실제 로직이 포함된 index.js 코드를 순식간에 생성해 주는 것을 확인하세요. 이를 통해 복잡한 백엔드 설정 없이도, 자연어만으로 서버 코드를 만들 수 있다는 것을 경험하게 됩니다.